# D2 – Reem – Ingestion, Metadata, and Data Quality

**Owner:** Reem  
**Component:** D2 — Ingestion pipeline and evidence-quality notebook  
**Files owned:** `src/ingest/pdf_loader.py`, `src/ingest/chunker.py`, `src/ingest/metadata_loader.py`, `src/ingest/run_ingest.py`

### What this notebook proves
1. Number of PDFs processed
2. Total chunk count + chunk-length statistics
3. Metadata completeness across required fields
4. Page-map examples (page number preserved per chunk)
5. Three chunk examples with full document + page provenance
6. Ingestion quality issues and fixes
7. Final summary saved to `reports/tables/d2_ingestion_summary.csv`


## Cell 1 — Imports and Project Paths

In [9]:
import sys
from pathlib import Path
import statistics
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.ingest.pdf_loader import load_pdf_pages
from src.ingest.chunker import chunk_pages
from src.ingest.metadata_loader import load_metadata, CLIMATE_METADATA_FIELDS

METADATA_CSV = PROJECT_ROOT / 'data' / 'metadata' / 'papers_metadata.csv'
PDF_DIR      = PROJECT_ROOT / 'data' / 'pdfs'
REPORTS_DIR  = PROJECT_ROOT / 'reports' / 'tables'
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('Metadata CSV :', METADATA_CSV, '| exists:', METADATA_CSV.exists())
print('PDF dir      :', PDF_DIR,      '| exists:', PDF_DIR.exists())


Project root : /Users/reembinhaider/Documents/GitHub/climate_evidence_graphrag_agent
Metadata CSV : /Users/reembinhaider/Documents/GitHub/climate_evidence_graphrag_agent/data/metadata/papers_metadata.csv | exists: True
PDF dir      : /Users/reembinhaider/Documents/GitHub/climate_evidence_graphrag_agent/data/pdfs | exists: True


## Cell 2 — Load Metadata and Count PDFs

In [10]:
records = load_metadata(METADATA_CSV)
print(f'Metadata rows loaded : {len(records)}')

present, missing_pdf = [], []
for rec in records:
    pdf_path = PROJECT_ROOT / rec['pdf_path']
    if pdf_path.exists():
        rec['_abs_pdf'] = pdf_path
        present.append(rec)
    else:
        missing_pdf.append(rec['document_id'])

print(f'PDFs found on disk   : {len(present)}')
print(f'PDFs missing on disk : {len(missing_pdf)}')
if missing_pdf:
    print('  First 5 missing:', missing_pdf[:5])

pd.DataFrame(records[:5])[['document_id','title','year','total_pages','topics']]


Metadata rows loaded : 300
PDFs found on disk   : 300
PDFs missing on disk : 0


,document_id,title,year,total_pages,topics
0,cuzzolin_2023_reasoning_random_sets_agenda_fut...,Reasoning with random sets: An agenda for the ...,2023,94,climate AI; policy and governance
1,staffell_2018_role_hydrogen_fuel_cells_global_...,The role of hydrogen and fuel cells in the glo...,2018,29,carbon capture; climate science; energy transi...
2,friedlingstein_2020_global_carbon_budget_2020_...,Global Carbon Budget 2020,2020,72,
3,friedlingstein_2022_global_carbon_budget_2022_...,Global Carbon Budget 2022,2022,90,climate science
4,bl_schl_2019_changing_climate_both_increases_d...,Changing climate both increases and decreases ...,2019,24,policy and governance; renewable energy


## Cell 3 — Run Pipeline: PDF → Pages → Chunks

In [11]:
# chunk_size=600 chars captures 1-2 complete climate claims per chunk.
# overlap=80 chars preserves sentences that fall on chunk boundaries.
CHUNK_SIZE = 600
OVERLAP    = 80

all_chunks, page_counts, skipped = [], [], []

for rec in present:
    try:
        pages  = load_pdf_pages(rec['_abs_pdf'], rec['document_id'])
        chunks = chunk_pages(pages, chunk_size=CHUNK_SIZE, overlap=OVERLAP)
        for c in chunks:
            c.title         = rec.get('title', '')
            c.year          = rec.get('year', '')
            c.topics        = rec.get('topics', '')
            c.countries     = rec.get('countries', '')
            c.sectors       = rec.get('sectors', '')
            c.climate_risks = rec.get('climate_risks', '')
        all_chunks.extend(chunks)
        page_counts.append(len(pages))
    except Exception as exc:
        skipped.append({'document_id': rec['document_id'], 'error': str(exc)})

total_pages  = sum(page_counts)
total_chunks = len(all_chunks)

print(f'Documents processed  : {len(present) - len(skipped)}')
print(f'Documents skipped    : {len(skipped)}')
print(f'Total pages extracted: {total_pages}')
print(f'Total chunks created : {total_chunks}')
if page_counts:
    print(f'Pages/doc  min={min(page_counts)}  median={statistics.median(page_counts):.0f}  max={max(page_counts)}')


Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set non-stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported
Cannot set stroke color: 2 components specified, but only 1 (grayscale), 3 (RGB), and 4 (CMYK) are supported

Documents processed  : 300
Documents skipped    : 0
Total pages extracted: 7588
Total chunks created : 57939
Pages/doc  min=3  median=19  max=216


## Cell 4 — Chunk-Length Statistics

In [12]:
lengths = [len(c.text) for c in all_chunks]
if lengths:
    print(f'Chunk length (chars)')
    print(f'  min    : {min(lengths)}')
    print(f'  median : {statistics.median(lengths):.0f}')
    print(f'  mean   : {sum(lengths)/len(lengths):.0f}')
    print(f'  max    : {max(lengths)}')
else:
    print('No chunks produced.')


Chunk length (chars)
  min    : 1
  median : 600
  mean   : 555
  max    : 600


## Cell 5 — Page-Map Examples

In [13]:
for rec in present[:3]:
    pages = load_pdf_pages(rec['_abs_pdf'], rec['document_id'])
    empty = [p.page_number for p in pages if not p.text.strip()]
    print(f'doc: {rec["document_id"]}')
    print(f'  total pages : {len(pages)}')
    print(f'  empty pages : {len(empty)} {empty[:5] if empty else "(none - good)"}')
    if pages:
        preview = pages[0].text[:120].strip().replace('\n', ' ')
        print(f'  page 1 text : {preview!r}')
    print()


doc: cuzzolin_2023_reasoning_random_sets_agenda_future_2401_09435v1
  total pages : 94
  empty pages : 0 (none - good)
  page 1 text : 'Reasoning with random sets: An agenda for the future FabioCuzzolin VisualArtificialIntelligenceLaboratory(VAIL) OxfordBr'

doc: staffell_2018_role_hydrogen_fuel_cells_global_energy_system_w2905218447
  total pages : 29
  empty pages : 0 (none - good)
  page 1 text : 'Energy & Environmental Science REVIEW The role of hydrogen and fuel cells in the global energy system Citethis:EnergyEnv'

doc: friedlingstein_2020_global_carbon_budget_2020_w3093432062
  total pages : 72
  empty pages : 0 (none - good)
  page 1 text : 'EarthSyst.Sci.Data,12,3269–3340,2020 https://doi.org/10.5194/essd-12-3269-2020 ©Author(s)2020.Thisworkisdistributedunder'



## Cell 6 — Three Chunk Provenance Examples

In [14]:
if len(all_chunks) >= 3:
    sample_idx = [0, len(all_chunks) // 2, len(all_chunks) - 1]
    for rank, i in enumerate(sample_idx, 1):
        c = all_chunks[i]
        print('=' * 65)
        print(f'Example {rank}')
        print(f'  chunk_id    : {c.chunk_id}')
        print(f'  document_id : {c.document_id}')
        print(f'  title       : {getattr(c, "title", "")}')
        print(f'  page range  : {c.start_page} - {c.end_page}')
        print(f'  topics      : {getattr(c, "topics", "")}')
        print(f'  text (200 chars): {c.text[:200].strip()}')
    print('=' * 65)
else:
    print('Not enough chunks for 3 examples.')


Example 1
  chunk_id    : cuzzolin_2023_reasoning_random_sets_agenda_future_2401_09435v1_p1_c0
  document_id : cuzzolin_2023_reasoning_random_sets_agenda_future_2401_09435v1
  title       : Reasoning with random sets: An agenda for the future
  page range  : 1 - 1
  topics      : climate AI; policy and governance
  text (200 chars): Reasoning with random sets: An agenda for the future FabioCuzzolin VisualArtificialIntelligenceLaboratory(VAIL) OxfordBrookesUniversity,Oxford,UK fabio.cuzzolin@brookes.ac.uk January19,2024 Abstract I
Example 2
  chunk_id    : smith_2023_climate_uncertainty_impacts_optimal_mitigation_pathways_social_cost_2304_08957v3_p1_c5
  document_id : smith_2023_climate_uncertainty_impacts_optimal_mitigation_pathways_social_cost_2304_08957v3
  title       : Climate uncertainty impacts on optimal mitigation pathways and social cost of carbon
  page range  : 1 - 1
  topics      : climate science; mitigation; policy and governance
  text (200 chars): limate and climate unc

## Cell 7 — Metadata Completeness Audit

In [15]:
df_meta = pd.read_csv(METADATA_CSV, encoding='utf-8-sig').fillna('')
total   = len(df_meta)
check_fields = [
    'document_id','title','authors','year','pdf_path',
    'topics','countries','sectors','climate_risks','total_pages'
]
rows = []
for field in check_fields:
    if field in df_meta.columns:
        empty  = int((df_meta[field].astype(str).str.strip() == '').sum())
        pct    = f'{100*empty/total:.1f}%'
        status = 'OK' if empty == 0 else ('WARN' if empty < total * 0.2 else 'RISK')
    else:
        empty, pct, status = total, '100%', 'MISSING COLUMN'
    rows.append({'field': field, 'missing_count': empty, 'missing_%': pct, 'status': status})

df_audit = pd.DataFrame(rows)
print(f'Metadata audit across {total} records:')
print(df_audit.to_string(index=False))


Metadata audit across 300 records:
        field  missing_count missing_% status
  document_id              0      0.0%     OK
        title              0      0.0%     OK
      authors              0      0.0%     OK
         year              0      0.0%     OK
     pdf_path              0      0.0%     OK
       topics             21      7.0%   WARN
    countries             28      9.3%   WARN
      sectors             21      7.0%   WARN
climate_risks            130     43.3%   RISK
  total_pages              0      0.0%     OK


## Cell 8 — Ingestion Quality Issues and Fixes

In [16]:
issues = [
    {'issue': 'CSV missing doi and license columns',
     'file': 'src/ingest/metadata_loader.py',
     'fix': 'Moved doi/license to OPTIONAL_FIELDS; alias doi_or_url -> doi automatically',
     'status': 'Fixed'},
    {'issue': 'Scanned PDF pages crash extract_text()',
     'file': 'src/ingest/pdf_loader.py',
     'fix': 'Wrapped per-page extraction in try/except; empty pages kept in page map',
     'status': 'Fixed'},
    {'issue': 'Corrupted PDF crashes entire pipeline',
     'file': 'src/ingest/run_ingest.py',
     'fix': 'Added try/except around load+chunk; logs ERROR and continues',
     'status': 'Fixed'},
]
for s in skipped:
    issues.append({'issue': f'PDF failed to load: {s["document_id"]}',
                   'file': 'data/pdfs/', 'fix': s['error'], 'status': 'Skipped (logged)'})

df_issues = pd.DataFrame(issues)
print(df_issues.to_string(index=False))


                                 issue                          file                                                                         fix status
   CSV missing doi and license columns src/ingest/metadata_loader.py Moved doi/license to OPTIONAL_FIELDS; alias doi_or_url -> doi automatically  Fixed
Scanned PDF pages crash extract_text()      src/ingest/pdf_loader.py     Wrapped per-page extraction in try/except; empty pages kept in page map  Fixed
 Corrupted PDF crashes entire pipeline      src/ingest/run_ingest.py                Added try/except around load+chunk; logs ERROR and continues  Fixed


## Cell 9 — Save Ingestion Summary CSV

In [17]:
lengths = [len(c.text) for c in all_chunks]
summary = [
    {'metric': 'metadata_rows_loaded',    'value': len(records),    'notes': 'from papers_metadata.csv'},
    {'metric': 'pdfs_found_on_disk',      'value': len(present),    'notes': f'out of {len(records)} records'},
    {'metric': 'pdfs_missing_on_disk',    'value': len(missing_pdf),'notes': 'pdf_path not found'},
    {'metric': 'pdfs_skipped_load_error', 'value': len(skipped),    'notes': 'pdfplumber error'},
    {'metric': 'total_pages_extracted',   'value': total_pages,     'notes': 'summed across all loaded PDFs'},
    {'metric': 'total_chunks_created',    'value': total_chunks,    'notes': f'chunk_size={CHUNK_SIZE} overlap={OVERLAP}'},
    {'metric': 'chunk_length_min',        'value': min(lengths) if lengths else 0,                  'notes': 'chars'},
    {'metric': 'chunk_length_median',     'value': int(statistics.median(lengths)) if lengths else 0,'notes': 'chars'},
    {'metric': 'chunk_length_max',        'value': max(lengths) if lengths else 0,                  'notes': 'chars'},
]
out = REPORTS_DIR / 'd2_ingestion_summary.csv'
pd.DataFrame(summary).to_csv(out, index=False)
print(f'Saved to {out}')
pd.DataFrame(summary)


Saved to /Users/reembinhaider/Documents/GitHub/climate_evidence_graphrag_agent/reports/tables/d2_ingestion_summary.csv


,metric,value,notes
0,metadata_rows_loaded,300,from papers_metadata.csv
1,pdfs_found_on_disk,300,out of 300 records
2,pdfs_missing_on_disk,0,pdf_path not found
3,pdfs_skipped_load_error,0,pdfplumber error
4,total_pages_extracted,7588,summed across all loaded PDFs
5,total_chunks_created,57939,chunk_size=600 overlap=80
6,chunk_length_min,1,chars
7,chunk_length_median,600,chars
8,chunk_length_max,600,chars


## Reem Reflection

**1. Why is chunk_size=600 chars reasonable for climate evidence retrieval?**  
I went with 600 chars because climate papers usually make their key point in one or two sentences — things like a specific temperature value or emissions figure. If the chunk is too big, that sentence gets lost in unrelated text and the retrieval score drops. If it is too small, you risk cutting a sentence in half and losing the meaning. The 80-char overlap was my way of making sure sentences that land right on a boundary still show up complete in at least one chunk.

**2. Which metadata field is most important for provenance, and why?**  
For me it is the page_number. The document_id tells you which paper, but without the page number you cannot actually go back and verify the claim in the original PDF. That is a problem for GraphRAG because citations need to point to a specific page, not just a document. I made sure every chunk keeps both so Rana and Alia can trace anything back to the source.

**3. What data-quality issue could weaken D2 results?**  
Scanned PDFs are the main risk. When a page is just an image, pdfplumber returns empty text, so that page produces zero chunks. The page is still in the map so the numbering stays correct, but if a whole paper is scanned it basically disappears from retrieval. I flagged this in the quality table — you can spot it by comparing the chunk count to total_pages in the metadata. A paper with 20 pages and 0 chunks is almost definitely scanned.
